# 01: Data Loading and Classification

This notebook handles:
- Loading the LLPS protein dataset
- Classifying proteins by pLLPS threshold
- Identifying membrane proteins
- Saving results for downstream analysis

**Outputs:**
- `results/full_dataset.csv` - Complete dataset with classifications
- `results/high_pllps_proteins.csv` - High pLLPS proteins (≥0.7)
- `results/membrane_proteins.csv` - Membrane proteins
- `results/classification_summary.json` - Summary statistics

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import llps_functions as lf

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
# Load the LLPS dataset
df = lf.load_llps_data('Human Phase separation data.xlsx')
print(f"\n📊 Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Display first few rows
df.head()

## 2. Classify Proteins by pLLPS Threshold

In [ ]:
# Set classification thresholds
HIGH_THRESHOLD = 0.7
LOW_THRESHOLD = 0.4

# Classify proteins
df_classified, high_pllps_ids = lf.get_high_pllps_proteins(
    df, 
    threshold=HIGH_THRESHOLD
)

print(f"\n✅ Classification complete:")
print(f"   High pLLPS (≥{HIGH_THRESHOLD}): {len(high_pllps_ids)} proteins")
print(f"   Percentage: {len(high_pllps_ids)/len(df)*100:.1f}%")

In [ ]:
# Add classification column
def classify_pllps(score):
    if pd.isna(score):
        return 'Unknown'
    elif score >= HIGH_THRESHOLD:
        return 'High'
    elif score <= LOW_THRESHOLD:
        return 'Low'
    else:
        return 'Medium'

df_classified['pLLPS_Class'] = df_classified['p(LLPS)'].apply(classify_pllps)

# Show class distribution
print("\n📊 pLLPS Class Distribution:")
print(df_classified['pLLPS_Class'].value_counts())
print(f"\nPercentages:")
print(df_classified['pLLPS_Class'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize pLLPS distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_classified['p(LLPS)'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(HIGH_THRESHOLD, color='red', linestyle='--', label=f'High threshold ({HIGH_THRESHOLD})')
axes[0].axvline(LOW_THRESHOLD, color='blue', linestyle='--', label=f'Low threshold ({LOW_THRESHOLD})')
axes[0].set_xlabel('p(LLPS)')
axes[0].set_ylabel('Count')
axes[0].set_title('pLLPS Distribution')
axes[0].legend()

# Box plot by class
df_classified.boxplot(column='p(LLPS)', by='pLLPS_Class', ax=axes[1])
axes[1].set_title('pLLPS by Classification')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('p(LLPS)')

plt.tight_layout()
plt.savefig('results/pllps_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved plot: results/pllps_distribution.png")

## 3. Identify Membrane Proteins

In [ ]:
# Filter to membrane proteins
membrane_df = lf.filter_membrane_proteins(df_classified)

print(f"\n📊 Membrane protein statistics:")
print(f"   Total membrane proteins: {len(membrane_df)}")
print(f"   High pLLPS membrane proteins: {(membrane_df['pLLPS_Class'] == 'High').sum()}")
print(f"   Medium pLLPS membrane proteins: {(membrane_df['pLLPS_Class'] == 'Medium').sum()}")
print(f"   Low pLLPS membrane proteins: {(membrane_df['pLLPS_Class'] == 'Low').sum()}")

In [ ]:
# Membrane vs non-membrane pLLPS comparison
df_classified['Is_Membrane'] = df_classified['Entry'].isin(membrane_df['Entry'])

fig, ax = plt.subplots(figsize=(10, 6))
df_classified.boxplot(column='p(LLPS)', by='Is_Membrane', ax=ax)
ax.set_title('pLLPS: Membrane vs Non-Membrane Proteins')
ax.set_xlabel('Is Membrane Protein')
ax.set_ylabel('p(LLPS)')
plt.tight_layout()
plt.savefig('results/membrane_vs_nonmembrane.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved plot: results/membrane_vs_nonmembrane.png")

## 4. Save Results

In [ ]:
# Save full classified dataset
lf.save_analysis_result(df_classified, 'full_dataset', format='csv')

# Save high pLLPS proteins
high_pllps_df = df_classified[df_classified['pLLPS_Class'] == 'High'].copy()
lf.save_analysis_result(high_pllps_df, 'high_pllps_proteins', format='csv')

# Save membrane proteins
lf.save_analysis_result(membrane_df, 'membrane_proteins', format='csv')

# Save summary statistics
summary = {
    'total_proteins': len(df_classified),
    'high_pllps_count': len(high_pllps_df),
    'high_pllps_percentage': len(high_pllps_df) / len(df_classified) * 100,
    'membrane_protein_count': len(membrane_df),
    'membrane_protein_percentage': len(membrane_df) / len(df_classified) * 100,
    'high_pllps_membrane_count': (membrane_df['pLLPS_Class'] == 'High').sum(),
    'thresholds': {
        'high': HIGH_THRESHOLD,
        'low': LOW_THRESHOLD
    },
    'class_distribution': df_classified['pLLPS_Class'].value_counts().to_dict()
}

lf.save_analysis_result(summary, 'classification_summary', format='json')

print("\n" + "="*60)
print("✅ All results saved successfully!")
print("="*60)

In [ ]:
# List saved files
lf.list_saved_results()

## Summary

✅ **Completed:**
1. Loaded LLPS dataset
2. Classified proteins into High/Medium/Low pLLPS categories
3. Identified membrane proteins
4. Generated visualizations
5. Saved all results to `results/` directory

**Next step:** Run `02_string_interactions.ipynb` to fetch protein-protein interactions from STRING database.